# Loan Approval Dashboard

using the basis of an US loan approval dataset from Kaggle, synthetic dataset has been generated with the following attributes: 
person_age	   
person_gender  
person_education  
person_income  
person_emp_exp  
person_home_ownership  
loan_amnt  
loan_intent	loan_int_rate  
loan_percent_income  
cb_person_cred_hist_length  
credit_score  
previous_loan_defaults_on_file  
loan_status


In [97]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from urllib.parse import quote_plus
#from dotenv import load_dotenv
import functools, operator

In [142]:
data = pd.read_csv('au_loan_data.csv')
data


,person_age,person_gender,person_education,person_state,person_income,person_emp_status,person_emp_exp,person_home_ownership,has_help_debt,help_debt_amount,loan_amnt,loan_intent,loan_term_months,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status,application_date
0,50,Female,Doctorate,WA,53716,Unemployed,4,RENT,0,0,13300,HOME_IMPROVEMENT,60,17.32,0.2476,32,713,No,0,2023-06-15
1,41,Male,Certificate/Diploma,VIC,20000,Full-time,9,RENT,0,0,30200,DEBT_CONSOLIDATION,36,11.59,1.5100,22,845,No,0,2025-12-26
2,40,Male,Postgraduate,SA,34133,Casual,8,MORTGAGE,0,0,22200,PERSONAL,36,17.32,0.6504,20,755,No,1,2024-11-01
3,40,Female,Bachelor,WA,71808,Full-time,2,OWN,1,19626,15800,DEBT_CONSOLIDATION,60,10.51,0.2200,18,752,No,1,2022-08-05
4,71,Female,Bachelor,VIC,36560,Full-time,10,RENT,0,0,10600,DEBT_CONSOLIDATION,36,17.24,0.2899,51,655,No,0,2024-03-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44995,39,Male,Bachelor,NSW,79355,Part-time,9,MORTGAGE,1,23714,32700,VEHICLE,60,14.75,0.4121,18,814,No,1,2023-02-04
44996,75,Female,Certificate/Diploma,QLD,41076,Full-time,7,OWN,0,0,13300,DEBT_CONSOLIDATION,12,17.93,0.3238,55,688,Yes,0,2023-10-01
44997,32,Male,High School,NSW,40313,Part-time,4,FAMILY,0,0,10200,HOME_IMPROVEMENT,72,14.45,0.2530,12,905,No,1,2024-08-17
44998,49,Male,Certificate/Diploma,NSW,28757,Part-time,7,OWN,0,0,14300,VEHICLE,36,16.68,0.4973,30,658,No,0,2022-12-13


### Dataset overview:

In [143]:
data.shape

(45000, 20)

In [100]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  int64  
 1   person_gender                   45000 non-null  object 
 2   person_education                45000 non-null  object 
 3   person_state                    45000 non-null  object 
 4   person_income                   45000 non-null  int64  
 5   person_emp_status               45000 non-null  object 
 6   person_emp_exp                  45000 non-null  int64  
 7   person_home_ownership           45000 non-null  object 
 8   has_help_debt                   45000 non-null  int64  
 9   help_debt_amount                45000 non-null  int64  
 10  loan_amnt                       45000 non-null  int64  
 11  loan_intent                     45000 non-null  object 
 12  loan_term_months                

In [101]:
data.head()

,person_age,person_gender,person_education,person_state,person_income,person_emp_status,person_emp_exp,person_home_ownership,has_help_debt,help_debt_amount,loan_amnt,loan_intent,loan_term_months,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status,application_date
0,50,Female,Doctorate,WA,53716,Unemployed,4,RENT,0,0,13300,HOME_IMPROVEMENT,60,17.32,0.2476,32,713,No,0,2023-06-15
1,41,Male,Certificate/Diploma,VIC,20000,Full-time,9,RENT,0,0,30200,DEBT_CONSOLIDATION,36,11.59,1.5100,22,845,No,0,2025-12-26
2,40,Male,Postgraduate,SA,34133,Casual,8,MORTGAGE,0,0,22200,PERSONAL,36,17.32,0.6504,20,755,No,1,2024-11-01
3,40,Female,Bachelor,WA,71808,Full-time,2,OWN,1,19626,15800,DEBT_CONSOLIDATION,60,10.51,0.2200,18,752,No,1,2022-08-05
4,71,Female,Bachelor,VIC,36560,Full-time,10,RENT,0,0,10600,DEBT_CONSOLIDATION,36,17.24,0.2899,51,655,No,0,2024-03-31


In [102]:
data.describe()

,person_age,person_income,person_emp_exp,has_help_debt,help_debt_amount,loan_amnt,loan_term_months,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,loan_status
count,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000
mean,45.203067,69671.052511,7.902822,0.261289,5624.383333,19898.511111,44.193067,13.868166,0.380636,25.213200,750.014200,0.537956
std,12.964998,41039.779524,5.074021,0.439342,11867.780144,13531.481566,16.951981,3.482621,0.358898,13.041812,159.552923,0.498563
min,19.000000,20000.000000,0.000000,0.000000,0.000000,2000.000000,12.000000,5.500000,0.007500,0.000000,58.000000,0.000000
25%,35.000000,41341.500000,4.000000,0.000000,0.000000,10400.000000,36.000000,11.490000,0.152300,15.000000,642.000000,0.000000
50%,43.000000,60041.500000,7.000000,0.000000,0.000000,16200.000000,36.000000,13.860000,0.271000,23.000000,751.000000,1.000000
75%,53.000000,86726.250000,11.000000,1.000000,6552.000000,25300.000000,60.000000,16.230000,0.479200,33.000000,859.000000,1.000000
max,75.000000,450000.000000,42.000000,1.000000,120000.000000,75000.000000,84.000000,25.000000,3.750000,57.000000,1200.000000,1.000000


In [146]:
#clearing whitespace 
for col in  ['person_gender', 'person_education', 'person_home_ownership', 'loan_intent', 'previous_loan_defaults_on_file']:
    data[col] = data[col].astype(str).str.strip()

#changing column names to make it easierr to find later
data.rename(columns={'cb_person_cred_hist_length': 'credit_history_length'}, inplace=True)

### Pruning the dataset before filling in nulls:

In [145]:
# taking care of duplicate applications
print("Duplicate rows: ", data.duplicated().sum())
data = data.drop_duplicates()

Duplicate rows:  0


In [149]:
# Datatype validation

# integers first:
int_cols = ['person_age','person_emp_exp','loan_amnt','loan_status','credit_history_length','has_help_debt','help_debt_amount','credit_score','loan_term_months']

for col in int_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')

# floats
float_cols = ['person_income','loan_int_rate','loan_percent_income']

for col in float_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')

# dates
date_cols = ['application_date']

for col in date_cols:
    data[col] = pd.to_datetime(data[col], errors='coerce')

# categories
cat_cols = ['person_gender','person_education','person_state','person_emp_status','person_home_ownership','loan_intent','previous_loan_defaults_on_file']

for col in cat_cols:
    data[col] = (data[col].astype(str).str.strip().str.upper())

In [151]:
# verify datatypes
print(data.dtypes)

person_age                                 int64
person_gender                             object
person_education                          object
person_state                              object
person_income                              int64
person_emp_status                         object
person_emp_exp                             int64
person_home_ownership                     object
has_help_debt                              int64
help_debt_amount                           int64
loan_amnt                                  int64
loan_intent                               object
loan_term_months                           int64
loan_int_rate                            float64
loan_percent_income                      float64
credit_history_length                      int64
credit_score                               int64
previous_loan_defaults_on_file            object
loan_status                                int64
application_date                  datetime64[ns]
dtype: object


In [152]:
# null check after conversion
print(data.isnull().sum())

person_age                        0
person_gender                     0
person_education                  0
person_state                      0
person_income                     0
person_emp_status                 0
person_emp_exp                    0
person_home_ownership             0
has_help_debt                     0
help_debt_amount                  0
loan_amnt                         0
loan_intent                       0
loan_term_months                  0
loan_int_rate                     0
loan_percent_income               0
credit_history_length             0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
application_date                  0
dtype: int64


In [153]:
# drop only required nulls
ignore = ['loan_percent_income', 'loan_int_rate', 'loan_status', 'previous_loan_defaults_on_file']
included_cols = [col for col in data.columns if col not in ignore]
data.dropna(subset=included_cols, inplace=True)

In [154]:
#Invalid / Zero value loan requests
data = data[data['loan_amnt'] > 0]

In [155]:
#repayment term
data = data[data['loan_term_months'] > 0]

In [156]:
#credit score check acc to australian standards
data = data[(data['credit_score'] >0 ) & (data['credit_score'] < 1200)]

In [157]:
#credit history check
data = data[(data['credit_history_length']>=0) & (data['credit_history_length'] < 60) ]

In [158]:
#realistic age check
data = data[(data['person_age']>=18) & (data['person_age']<=100)]

In [159]:
#Invalid / Zero value income
data = data[(data['person_income'] > 0)]

In [160]:
# realistic experience check
data = data[(data['person_emp_exp']>=0) & (data['person_emp_exp']<=70)]

In [161]:
data = data[(data['help_debt_amount']>=0) & (data['help_debt_amount']<=150000)]

In [162]:
# allowed category values

valid_gender = ['MALE','FEMALE','NON-BINARY']

valid_education = ['HIGH SCHOOL','CERTIFICATE/DIPLOMA','BACHELOR','POSTGRADUATE','DOCTORATE']

valid_states = ['NSW','VIC','QLD','WA','SA','TAS','ACT','NT']

valid_emp_status = ['FULL-TIME','PART-TIME','SELF-EMPLOYED','CASUAL','UNEMPLOYED']

valid_home_ownership = ['RENT','MORTGAGE','OWN','FAMILY','OTHER']

valid_loan_intent = ['VEHICLE','DEBT_CONSOLIDATION','HOME_IMPROVEMENT','MEDICAL','HOLIDAY','EDUCATION','PERSONAL']

valid_defaults = ['YES','NO']

In [168]:
# filter out invalid categories
data = data[data['person_gender'].isin(valid_gender)]

In [169]:
data = data[data['person_education'].isin(valid_education)]

In [170]:
data = data[data['person_state'].isin(valid_states)]

In [171]:
data = data[data['person_emp_status'].isin(valid_emp_status)]

In [172]:
data = data[data['person_home_ownership'].isin(valid_home_ownership)]

In [173]:
data = data[data['loan_intent'].isin(valid_loan_intent)]

In [174]:
data = data[data['previous_loan_defaults_on_file'].isin(valid_defaults)]

In [175]:
data.shape

(44888, 20)

### Checking Nulls and filling data

In [176]:
data.isnull().sum()
#no null values because the dataset used is a synthetic one.
#still I've added checks for filling in null values

person_age                        0
person_gender                     0
person_education                  0
person_state                      0
person_income                     0
person_emp_status                 0
person_emp_exp                    0
person_home_ownership             0
has_help_debt                     0
help_debt_amount                  0
loan_amnt                         0
loan_intent                       0
loan_term_months                  0
loan_int_rate                     0
loan_percent_income               0
credit_history_length             0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
application_date                  0
dtype: int64

In [177]:
#Interest rate check
if data['loan_int_rate'].isnull().sum() > 0:
    data['loan_int_rate'].fillna((data.groupby(['loan_intent', 'home_ownership', ])['loan_int_rate'].median()), inplace=True)
data = data[(data['loan_int_rate'] > 5.0) & (data['loan_int_rate'] <30.0)]

In [178]:
#Loan status check (0: rejected 1: accepted 2: pending)
if data['loan_status'].isnull().sum() > 0:
    data['loan_status'].fillna(2, inplace=True)
data = data[data['loan_status'].isin([0,1,2])]

In [179]:
data.head(5)

,person_age,person_gender,person_education,person_state,person_income,person_emp_status,person_emp_exp,person_home_ownership,has_help_debt,help_debt_amount,loan_amnt,loan_intent,loan_term_months,loan_int_rate,loan_percent_income,credit_history_length,credit_score,previous_loan_defaults_on_file,loan_status,application_date
0,50,FEMALE,DOCTORATE,WA,53716,UNEMPLOYED,4,RENT,0,0,13300,HOME_IMPROVEMENT,60,17.32,0.2476,32,713,NO,0,2023-06-15
1,41,MALE,CERTIFICATE/DIPLOMA,VIC,20000,FULL-TIME,9,RENT,0,0,30200,DEBT_CONSOLIDATION,36,11.59,1.5100,22,845,NO,0,2025-12-26
2,40,MALE,POSTGRADUATE,SA,34133,CASUAL,8,MORTGAGE,0,0,22200,PERSONAL,36,17.32,0.6504,20,755,NO,1,2024-11-01
3,40,FEMALE,BACHELOR,WA,71808,FULL-TIME,2,OWN,1,19626,15800,DEBT_CONSOLIDATION,60,10.51,0.2200,18,752,NO,1,2022-08-05
4,71,FEMALE,BACHELOR,VIC,36560,FULL-TIME,10,RENT,0,0,10600,DEBT_CONSOLIDATION,36,17.24,0.2899,51,655,NO,0,2024-03-31


### Derived values

In [180]:
#Loan to income ratio recalculation since we dropped some values
data['loan_percent_income'] = data['loan_amnt'] / data['person_income'].round(2)

In [181]:
#Income tiers
data['income_tier'] = pd.cut(data['person_income'], bins = [0, 50000, 100000, 200000, 500000], labels=['LOW', 'MIDDLE', 'UPPER-MIDDLE', 'HIGH'])

In [182]:
# income to loan ratio bands
data['dti_band'] = pd.cut(data['loan_percent_income'], bins = [0, 0.15, 0.35, 1.0, 5.0], labels=['LOW', 'MIDDLE', 'HIGH', 'VERY HIGH'])

In [183]:
# age groups
data['age_group'] = pd.cut(data['person_age'], bins = [18, 25, 35, 50, 100], labels=['YOUNG ADULT', 'ADULT', 'MIDDLE-AGED', 'SENIOR'])

In [184]:
# credit score categories
data['credit_score_category'] = pd.cut(data['credit_score'], bins = [0, 460, 661, 735, 853, 1200], labels=['BELOW AVG', 'AVG', 'GOOD', 'VERY GOOD', 'EXCELLENT'])

In [185]:
#leanding risk flags

#has existing help debt + low income (error somewhere)
#data['flag_help_debt'] = ((data['has_help_debt'] == 1) & (data['income_tier'] == 'LOW')).astype(int) 

#young applicant + large loan request
data['flag_young_high_loan'] = ((data['person_age'] <25) & (data['loan_amnt'] > 50000)).astype(int)

In [186]:
#risk flags contd

#very high loan request compared to income
data['flag_income_loan_mismatch'] = (data['loan_percent_income'] > 0.5).astype(int)

# Employment experience impossible given age
data['flag_impossible_exp'] = (data['person_emp_exp'] >= (data['person_age'] - 16)).astype(int)

# Credit history longer than working life
data['flag_impossible_credit_history'] = (data['credit_history_length'] >= data['person_age'] - 16).astype(int)

# Suspiciously round high income (common fraudulent entries)
data['flag_round_income'] = ((data['person_income'] % 10000 == 0) & (data['person_income'] > 100000)).astype(int)

#previously defaulted but approved now
data['flag_default'] = ((data['previous_loan_defaults_on_file']=='YES') & (data['loan_status']== 1)).astype(int)

#unemployed yet approved for loan
data['flag_unemployment'] = ((data['person_emp_status']=='UNEMPLOYED') & (data['loan_status']== 1)).astype(int)

flag_cols = [
    'flag_income_loan_mismatch',
    'flag_impossible_exp',
    'flag_impossible_credit_history',
    'flag_round_income',
    'flag_unemployment',
    'flag_default'
]

data['risk_flag_count'] = data[flag_cols].sum(axis=1)

In [187]:
#sort by loan application date
data = data.sort_values('application_date').reset_index(drop=True)

In [196]:
#id
data.insert(0, 'id', range(100000, len(data) + 100000))
data['id'] = ('APP' + data['id'].astype(str))

<class 'ValueError'>: cannot insert id, already exists

In [195]:
data.head(5)

,id,person_age,person_gender,person_education,person_state,person_income,person_emp_status,person_emp_exp,person_home_ownership,has_help_debt,...,age_group,credit_score_category,flag_young_high_loan,flag_income_loan_mismatch,flag_impossible_exp,flag_impossible_credit_history,flag_round_income,flag_default,flag_unemployment,risk_flag_count
0,APP100000,39,MALE,HIGH SCHOOL,QLD,62994,FULL-TIME,9,OWN,0,...,MIDDLE-AGED,VERY GOOD,0,0,0,0,0,0,0,0
1,APP100001,25,MALE,CERTIFICATE/DIPLOMA,SA,77107,FULL-TIME,5,MORTGAGE,0,...,YOUNG ADULT,GOOD,0,0,0,0,0,0,0,0
2,APP100002,45,FEMALE,CERTIFICATE/DIPLOMA,WA,30752,PART-TIME,21,RENT,0,...,MIDDLE-AGED,GOOD,0,1,0,0,0,0,0,1
3,APP100003,38,MALE,BACHELOR,VIC,84232,FULL-TIME,5,FAMILY,1,...,MIDDLE-AGED,EXCELLENT,0,1,0,0,0,0,0,1
4,APP100004,75,MALE,BACHELOR,NSW,141352,PART-TIME,1,MORTGAGE,0,...,SENIOR,VERY GOOD,0,0,0,0,0,1,0,1


### Final look over

In [94]:
data.describe()

,person_age,person_income,person_emp_exp,has_help_debt,help_debt_amount,loan_amnt,loan_term_months,loan_int_rate,loan_percent_income,credit_history_length,...,loan_status,application_date,flag_young_high_loan,flag_income_loan_mismatch,flag_impossible_exp,flag_impossible_credit_history,flag_round_income,flag_default,flag_unemployment,risk_flag_count
count,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [197]:
data['loan_status'].value_counts()

loan_status
1    24129
0    20759
Name: count, dtype: int64

In [198]:
data.groupby('person_state')['loan_status'].mean()

person_state
ACT    0.524968
NSW    0.538831
NT     0.531732
QLD    0.536711
SA     0.534678
TAS    0.550214
VIC    0.537872
WA     0.536699
Name: loan_status, dtype: float64

In [199]:
data.groupby('income_tier')['loan_status'].mean()

<ipython-input-199-df8975feb920>:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data.groupby('income_tier')['loan_status'].mean()


income_tier
LOW             0.413213
MIDDLE          0.575853
UPPER-MIDDLE    0.694547
HIGH            0.777236
Name: loan_status, dtype: float64

### Final Validation

In [200]:
print(data.shape)

(44888, 33)


In [201]:
print(data.isnull().sum())

id                                0
person_age                        0
person_gender                     0
person_education                  0
person_state                      0
person_income                     0
person_emp_status                 0
person_emp_exp                    0
person_home_ownership             0
has_help_debt                     0
help_debt_amount                  0
loan_amnt                         0
loan_intent                       0
loan_term_months                  0
loan_int_rate                     0
loan_percent_income               0
credit_history_length             0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
application_date                  0
income_tier                       0
dti_band                          0
age_group                         0
credit_score_category             0
flag_young_high_loan              0
flag_income_loan_mismatch         0
flag_impossible_exp         

In [202]:
print(data.duplicated().sum())

0


### Exporting CSVs

In [203]:
loan_details = data[['id', 'application_date', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'loan_term_months', 'loan_status', 'dti_band']]

In [204]:
applicant_details = data[
    [
        'id',
        'person_age',
        'person_gender',
        'person_education',
        'person_income',
        'income_tier',
        'person_emp_status',
        'person_emp_exp',
        'person_home_ownership',
        'person_state'
    ]
]

In [205]:
credit_details = data[
    [
        'id',
        'credit_score',
        'credit_history_length',
        'previous_loan_defaults_on_file',
        'has_help_debt',
        'help_debt_amount'
    ]
]

In [ ]:
risk_flags = data[
    [
        'id', 
        'flag_income_loan_mismatch',
        'flag_impossible_exp',
        'flag_impossible_credit_history', 
        'flag_round_income', 
        'flag_default', 
        'flag_unemployment',  
        'flag_impossible_exp',
        'flag_unemployment',
        'risk_flag_count',
        'flag_young_high_loan' 
    ]
]

### Created CSV checks